In [0]:
from pyspark.sql.functions import col, sum
from pyspark.sql.functions import lower, count, when, round

In [0]:
# Leer el archivo CSV desde DBFS
transactions = spark.read.table("hive_metastore.default.raw_transactions")

transactions.limit(50).display()

User,Card,Year,Month,Day,Time,Amount,UseChip,MerchantName,MerchantCity,MerchantState,Zip,MCC,Errors,IsFraud?
0,0,2002,9,1,2024-09-11T06:21:00Z,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,null,No
0,0,2002,9,1,2024-09-11T06:42:00Z,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,null,No
0,0,2002,9,2,2024-09-11T06:22:00Z,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,null,No
0,0,2002,9,2,2024-09-11T17:45:00Z,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,null,No
0,0,2002,9,3,2024-09-11T06:23:00Z,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,null,No
0,0,2002,9,3,2024-09-11T13:53:00Z,$86.19,Swipe Transaction,-7146670748125200898,Monterey Park,CA,91755.0,5970,null,No
0,0,2002,9,4,2024-09-11T05:51:00Z,$93.84,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,null,No
0,0,2002,9,4,2024-09-11T06:09:00Z,$123.50,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,null,No
0,0,2002,9,5,2024-09-11T06:14:00Z,$61.72,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,null,No
0,0,2002,9,5,2024-09-11T09:35:00Z,$57.10,Swipe Transaction,4055257078481058705,La Verne,CA,91750.0,7538,null,No


In [0]:
transactions.printSchema()

root
 |-- User: integer (nullable = true)
 |-- Card: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Amount: string (nullable = true)
 |-- UseChip: string (nullable = true)
 |-- MerchantName: long (nullable = true)
 |-- MerchantCity: string (nullable = true)
 |-- MerchantState: string (nullable = true)
 |-- Zip: double (nullable = true)
 |-- MCC: integer (nullable = true)
 |-- Errors: string (nullable = true)
 |-- IsFraud?: string (nullable = true)



In [0]:
#Porcentaje de valores nulos en cada columna
filas = transactions.count()
print(filas)
transactions.select([round((count(when(col(c).isNull(), c))/filas),3).alias(c) for c in transactions.columns]).show()

24386900
+----+----+----+-----+---+----+------+-------+------------+------------+-------------+-----+---+------+--------+
|User|Card|Year|Month|Day|Time|Amount|UseChip|MerchantName|MerchantCity|MerchantState|  Zip|MCC|Errors|IsFraud?|
+----+----+----+-----+---+----+------+-------+------------+------------+-------------+-----+---+------+--------+
| 0.0| 0.0| 0.0|  0.0|0.0| 0.0|   0.0|    0.0|         0.0|         0.0|        0.112|0.118|0.0| 0.984|     0.0|
+----+----+----+-----+---+----+------+-------+------------+------------+-------------+-----+---+------+--------+



### Corrección de nombres de columnas

In [0]:
# Eliminar ? de la columnaIsFraud
transactions = transactions.withColumnRenamed("IsFraud?", "IsFraud")
transactions.show(1)

+----+----+----+-----+---+-------------------+-------+-----------------+-------------------+------------+-------------+-------+----+------+-------+
|User|Card|Year|Month|Day|               Time| Amount|          UseChip|       MerchantName|MerchantCity|MerchantState|    Zip| MCC|Errors|IsFraud|
+----+----+----+-----+---+-------------------+-------+-----------------+-------------------+------------+-------------+-------+----+------+-------+
|   0|   0|2002|    9|  1|2024-09-11 06:21:00|$134.09|Swipe Transaction|3527213246127876953|    La Verne|           CA|91750.0|5300|  NULL|     No|
+----+----+----+-----+---+-------------------+-------+-----------------+-------------------+------------+-------------+-------+----+------+-------+
only showing top 1 row



### Columna Errors a binario

In [0]:
# ¿? Cambio de Valor nulo en la columna error por "No"
transactions = transactions.fillna("0", subset=["Errors"])
transactions = transactions.withColumn("Errors", when(col("Errors") != "0" , 1).otherwise(0))
transactions.select("Errors").distinct().show()

+------+
|Errors|
+------+
|     1|
|     0|
+------+



### Correción de formatos de fechas

In [0]:
from pyspark.sql.functions import date_format, concat_ws, to_timestamp, hour

In [0]:
#Limpieza del formato de la columna Time
trs_horas = transactions.withColumn("Hour", date_format(transactions.time, "HH:mm:ss")).drop("Time")
trs_horas.show(2)

+----+----+----+-----+---+-------+-----------------+-------------------+-------------+-------------+-------+----+------+-------+--------+
|user|card|year|month|day| amount|          usechip|       merchantname| merchantcity|merchantstate|    zip| mcc|Errors|isfraud|    hour|
+----+----+----+-----+---+-------+-----------------+-------------------+-------------+-------------+-------+----+------+-------+--------+
|   0|   0|2002|    9|  1|$134.09|Swipe Transaction|3527213246127876953|     La Verne|           CA|91750.0|5300|     0|     No|06:21:00|
|   0|   0|2002|    9|  1| $38.48|Swipe Transaction|-727612092139916043|Monterey Park|           CA|91754.0|5411|     0|     No|06:42:00|
+----+----+----+-----+---+-------+-----------------+-------------------+-------------+-------------+-------+----+------+-------+--------+
only showing top 2 rows



In [0]:
#Conversión de la columna Time a timestamp
transactions = trs_horas.withColumn("Time", to_timestamp(col("Hour"), "HH:mm:ss"))

# Rangos de horas usando la función hour()
transactions = transactions.withColumn("RangoHoras", 
    when(hour(col("Time")).between(0, 2), "Medianoche")
    .when(hour(col("Time")).between(3, 6), "Madrugada")
    .when(hour(col("Time")).between(7, 9), "Mañana")
    .when(hour(col("Time")).between(10, 11), "Mañana-Tarde")
    .when(hour(col("Time")).between(12, 13), "Medio día")
    .when(hour(col("Time")).between(14, 16), "Día")
    .when(hour(col("Time")).between(17, 19), "Tarde")
    .when(hour(col("Time")).between(20, 22), "Tarde-Noche")
    .otherwise("Noche"))

In [0]:
transactions.select("RangoHoras").distinct().show()

+------------+
| rango_horas|
+------------+
|       Noche|
| Tarde-Noche|
|   Madrugada|
|  Medianoche|
|   Medio día|
|      Mañana|
|         Día|
|       Tarde|
|Mañana-Tarde|
+------------+



### Conversión de horas a momento del día

In [0]:
# Normalización concatenando las columnas año, mes y dia
# Se agrega la columna DiaDeSemana con el fin de reducir el rango de fechas a tomar en cuenta
transactions = (
            transactions
            .withColumn('Date', concat_ws('-', col("Year"), col("Month"), col("Day")).cast('date'))
            .withColumn("DiaDeSemana", date_format(col("Date"), "EEEE"))
            .drop("Year", "Month", "Day","Hour", "Time", "Date"))


In [0]:
transactions.display()

user,card,amount,usechip,merchantname,merchantcity,merchantstate,zip,mcc,Errors,isfraud,rango_horas,dia_de_semana
591,3,$-415.00,Chip Transaction,4552887027432897467,Oakland,CA,94606.0,3596,0,No,Mañana-Tarde,Saturday
591,3,$22.37,Chip Transaction,-8964802287130046767,Tucker,GA,30084.0,7230,0,No,Medio día,Tuesday
591,3,$10.87,Chip Transaction,97032797689821735,Southern Pines,NC,28387.0,5411,0,No,Día,Friday
591,3,$73.84,Chip Transaction,-5401953891366957779,Shannon,NC,28386.0,5651,0,No,Mañana-Tarde,Saturday
591,3,$38.50,Chip Transaction,-2472481739355111587,Saint Pauls,NC,28384.0,7538,0,No,Día,Saturday
591,3,$10.54,Chip Transaction,3397452747792109772,Tucker,GA,30084.0,5541,0,No,Día,Thursday
591,3,$35.48,Swipe Transaction,3952145593743244256,Tucker,GA,30084.0,7538,0,No,Día,Sunday
591,3,$42.29,Swipe Transaction,-8278120102146349383,Ridgway,CO,81432.0,7538,0,No,Día,Friday
591,3,$13.19,Chip Transaction,4722913068560264812,Ridgway,CO,81432.0,5411,0,No,Día,Friday
591,3,$17.46,Online Transaction,-5841929396161652653,ONLINE,null,null,4121,0,No,Tarde-Noche,Saturday


### Use Chip normalizado

In [0]:
transactions.select("UseChip").distinct().show()

+------------------+
|           UseChip|
+------------------+
| Swipe Transaction|
|  Chip Transaction|
|Online Transaction|
+------------------+



In [0]:
transaction = transactions.withColumn("UseChip", when(col("UseChip") == "Swipe Transaction", 1).when(col("UseChip") == "Chip Transaction", 2).otherwise(0))
# Swipe Transaction = 1
# Chip Transaction = 2
# Online Transaction = 0
transaction.display(10)

user,card,amount,UseChip,merchantname,merchantcity,merchantstate,zip,mcc,Errors,isfraud,rango_horas,dia_de_semana
591,3,$-415.00,2,4552887027432897467,Oakland,CA,94606.0,3596,0,No,Mañana-Tarde,Saturday
591,3,$22.37,2,-8964802287130046767,Tucker,GA,30084.0,7230,0,No,Medio día,Tuesday
591,3,$10.87,2,97032797689821735,Southern Pines,NC,28387.0,5411,0,No,Día,Friday
591,3,$73.84,2,-5401953891366957779,Shannon,NC,28386.0,5651,0,No,Mañana-Tarde,Saturday
591,3,$38.50,2,-2472481739355111587,Saint Pauls,NC,28384.0,7538,0,No,Día,Saturday
591,3,$10.54,2,3397452747792109772,Tucker,GA,30084.0,5541,0,No,Día,Thursday
591,3,$35.48,1,3952145593743244256,Tucker,GA,30084.0,7538,0,No,Día,Sunday
591,3,$42.29,1,-8278120102146349383,Ridgway,CO,81432.0,7538,0,No,Día,Friday
591,3,$13.19,2,4722913068560264812,Ridgway,CO,81432.0,5411,0,No,Día,Friday
591,3,$17.46,0,-5841929396161652653,ONLINE,null,null,4121,0,No,Tarde-Noche,Saturday


In [0]:
transaction.select("UseChip").distinct().show()

+-------+
|UseChip|
+-------+
|      1|
|      2|
|      0|
+-------+



# Is fraud

In [0]:
transaction = transaction.withColumn("IsFraud", when(col("IsFraud")=="No",0).otherwise(1).cast('integer'))
transaction.limit(50).display()

user,card,amount,UseChip,merchantname,merchantcity,merchantstate,zip,mcc,Errors,isfraud,rango_horas,dia_de_semana
591,3,$-415.00,2,4552887027432897467,Oakland,CA,94606.0,3596,0,0,Mañana-Tarde,Saturday
591,3,$22.37,2,-8964802287130046767,Tucker,GA,30084.0,7230,0,0,Medio día,Tuesday
591,3,$10.87,2,97032797689821735,Southern Pines,NC,28387.0,5411,0,0,Día,Friday
591,3,$73.84,2,-5401953891366957779,Shannon,NC,28386.0,5651,0,0,Mañana-Tarde,Saturday
591,3,$38.50,2,-2472481739355111587,Saint Pauls,NC,28384.0,7538,0,0,Día,Saturday
591,3,$10.54,2,3397452747792109772,Tucker,GA,30084.0,5541,0,0,Día,Thursday
591,3,$35.48,1,3952145593743244256,Tucker,GA,30084.0,7538,0,0,Día,Sunday
591,3,$42.29,1,-8278120102146349383,Ridgway,CO,81432.0,7538,0,0,Día,Friday
591,3,$13.19,2,4722913068560264812,Ridgway,CO,81432.0,5411,0,0,Día,Friday
591,3,$17.46,0,-5841929396161652653,ONLINE,null,null,4121,0,0,Tarde-Noche,Saturday


# Merchanstate and zip

In [0]:
transaction = (
  transactions
  .withColumn("MerchantName", when(col("MerchantCity")=="ONLINE","ONLINE").otherwise(col("MerchantState")))
  .withColumn("Zip", when(col("MerchantCity")=="ONLINE", 00000).otherwise(col("Zip")).cast('integer'))
  )

transaction.limit(10).display()

user,card,amount,usechip,merchantname,merchantcity,merchantstate,zip,mcc,Errors,isfraud,rango_horas,dia_de_semana
591,3,$-415.00,Chip Transaction,4552887027432897467,Oakland,CA,94606,3596,0,No,Mañana-Tarde,Saturday
591,3,$22.37,Chip Transaction,-8964802287130046767,Tucker,GA,30084,7230,0,No,Medio día,Tuesday
591,3,$10.87,Chip Transaction,97032797689821735,Southern Pines,NC,28387,5411,0,No,Día,Friday
591,3,$73.84,Chip Transaction,-5401953891366957779,Shannon,NC,28386,5651,0,No,Mañana-Tarde,Saturday
591,3,$38.50,Chip Transaction,-2472481739355111587,Saint Pauls,NC,28384,7538,0,No,Día,Saturday
591,3,$10.54,Chip Transaction,3397452747792109772,Tucker,GA,30084,5541,0,No,Día,Thursday
591,3,$35.48,Swipe Transaction,3952145593743244256,Tucker,GA,30084,7538,0,No,Día,Sunday
591,3,$42.29,Swipe Transaction,-8278120102146349383,Ridgway,CO,81432,7538,0,No,Día,Friday
591,3,$13.19,Chip Transaction,4722913068560264812,Ridgway,CO,81432,5411,0,No,Día,Friday
591,3,$17.46,Online Transaction,-5841929396161652653,ONLINE,ONLINE,0,4121,0,No,Tarde-Noche,Saturday


### Amount a decimal

In [0]:
from pyspark.sql.functions import regexp_replace

In [0]:
transactions = (
    transaction
    .withColumn("Amount", regexp_replace("Amount", '\$', ' '))
    .withColumn("Amount", col("Amount").cast("float"))
)
transactions.show(10)

+----+----+------+------------------+--------------------+--------------+-------------+-----+----+------+-------+------------+-------------+
|user|card|amount|           usechip|        merchantname|  merchantcity|merchantstate|  zip| mcc|Errors|isfraud| rango_horas|dia_de_semana|
+----+----+------+------------------+--------------------+--------------+-------------+-----+----+------+-------+------------+-------------+
| 591|   3|-415.0|  Chip Transaction| 4552887027432897467|       Oakland|           CA|94606|3596|     0|     No|Mañana-Tarde|     Saturday|
| 591|   3| 22.37|  Chip Transaction|-8964802287130046767|        Tucker|           GA|30084|7230|     0|     No|   Medio día|      Tuesday|
| 591|   3| 10.87|  Chip Transaction|   97032797689821735|Southern Pines|           NC|28387|5411|     0|     No|         Día|       Friday|
| 591|   3| 73.84|  Chip Transaction|-5401953891366957779|       Shannon|           NC|28386|5651|     0|     No|Mañana-Tarde|     Saturday|
| 591|   3|  

## Tareas de Limpieza
- Eliminar el ? de Is Fraud
- Errores sustituir el valor nulo por "No" 
- Limpiar la columna Time dejando sólo la hora, sería 2024-09-11T20:41:00.000+00:00 -> 20:41
- convertir las columnas Year, Month y Day en una sola que sea Date y solo mantener el día de la semana numérico con el fin de reducir el rango de valores
- la columna UseChip cambiar valores únicos por número
- Manejar los nulos que ocasiona la columna MerchantCity en MerchantState y ZIP al ser compra ONLINE
- Para disminuir la cantidad de horas diferentes en la columna Time podemos establecer un rango de horas: Medianoche, madrugada, mañana, mañana-tarde, medio día, día, tarde, tarde noche
- MerchantName contiene muchos valores nulos como para eliminarla?
- Eliminar los registros con valores negativos de Amount?
- Después de esta limpieza ¿Utilizar grafos para analizarlo?
- Amount a Decimal!
- Merchant State, como tratar con los valores nulos de los registros ONLINE 
- ZIP, como tratar con los valores nulos de los registros ONLINE. ¿Tomamos el Zip como categorico o realizamos 

In [0]:
transactions.limit(50).display()

user,card,amount,usechip,merchantname,merchantcity,merchantstate,zip,mcc,Errors,isfraud,rango_horas,dia_de_semana
591,3,-415.0,Chip Transaction,4552887027432897467,Oakland,CA,94606,3596,0,No,Mañana-Tarde,Saturday
591,3,22.37,Chip Transaction,-8964802287130046767,Tucker,GA,30084,7230,0,No,Medio día,Tuesday
591,3,10.87,Chip Transaction,97032797689821735,Southern Pines,NC,28387,5411,0,No,Día,Friday
591,3,73.84,Chip Transaction,-5401953891366957779,Shannon,NC,28386,5651,0,No,Mañana-Tarde,Saturday
591,3,38.5,Chip Transaction,-2472481739355111587,Saint Pauls,NC,28384,7538,0,No,Día,Saturday
591,3,10.54,Chip Transaction,3397452747792109772,Tucker,GA,30084,5541,0,No,Día,Thursday
591,3,35.48,Swipe Transaction,3952145593743244256,Tucker,GA,30084,7538,0,No,Día,Sunday
591,3,42.29,Swipe Transaction,-8278120102146349383,Ridgway,CO,81432,7538,0,No,Día,Friday
591,3,13.19,Chip Transaction,4722913068560264812,Ridgway,CO,81432,5411,0,No,Día,Friday
591,3,17.46,Online Transaction,-5841929396161652653,ONLINE,ONLINE,0,4121,0,No,Tarde-Noche,Saturday


## Creación de la capa Staging

In [0]:
transactions.write.format('delta').mode('overwrite').saveAsTable('staging_transactions')